# Handcrafting the max circuit

Build a 2-block transformer that solves `c = max(a, b)` mod 97 **without any training** — weights placed by formula from the mechanism observed in Plan 6b. Verifies 100% accuracy on all 9409 (a, b) pairs and visualizes the resulting internal structure.

Mechanism (attention-copy):
- Block 0 head 0 does monotonic-in-value comparison: `Q · K` picks the position holding `max(a, b)`.
- The same head's V copies the chosen token's additive-Fourier coefficients into the residual at the `=` slot.
- The head reads those coefficients as a Dirichlet-kernel argmax over output classes.
- Block 1 and all FFNs are zero — they are passthrough by design.

Runs on CPU in seconds.

In [ ]:
# Cell 1 — setup
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = Path('.').resolve()

# Constants
P = 97
D_MODEL = 384
N_HEADS = 4
HEAD_DIM = D_MODEL // N_HEADS    # 96
N_BLOCKS = 2
VOCAB = P + 2                    # 99: tokens 0..96 + op (97) + eq (98)
SEQ_LEN = 4
K_FREQS = list(range(1, 49))     # 48 frequencies
K = len(K_FREQS)

# Layout (which dim carries what)
DIM_VAL, DIM_PAD, DIM_OP, DIM_EQ = 0, 1, 2, 3
POS_DIMS = (4, 5, 6, 7)
ADD_COS_BASE, ADD_SIN_BASE = 8, 56

# Tuned magnitudes
PAD_C, POS_M = 100.0, 100.0
Q_GAIN, K_GAIN, BIG = 100.0, 1.0, 1000.0

BG, FG, GRID = '#FAFAF7', '#2A2A2A', '#E5E3DD'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.6, 'axes.labelcolor': FG,
    'axes.titlecolor': FG, 'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.6,
})
print(f'P={P}, d={D_MODEL}, heads={N_HEADS}, K=1..{K_FREQS[-1]}')

In [ ]:
# Cell 2 — architecture (same as Plan 6 family minus dropout)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d)
        self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False)
        self.w3 = nn.Linear(d, 4*d, bias=False)
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + o
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        return x + self.w2(gate * self.w3(h2))

class GrokModelMax(nn.Module):
    def __init__(self, p=P, d=D_MODEL, nh=N_HEADS, n_blocks=N_BLOCKS):
        super().__init__()
        self.p, self.d = p, d
        self.tok_emb = nn.Embedding(p+2, d)
        self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh, dropout=0.0) for _ in range(n_blocks)])
        self.norm_f = RMSNorm(d)
        self.head = nn.Linear(d, p, bias=False)
    def forward(self, a, b):
        B = a.size(0)
        op = torch.full((B,), self.p, device=a.device, dtype=torch.long)
        eq = torch.full((B,), self.p+1, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        return self.head(self.norm_f(x)[:, -1, :])

In [ ]:
# Cell 3 — build the state dict from formulas, no training
def build_handcraft_max() -> dict:
    sd = {}
    # tok_emb: scalar value + Fourier of value + type markers
    W_E = torch.zeros(VOCAB, D_MODEL)
    for t in range(P):
        W_E[t, DIM_VAL] = float(t)
        W_E[t, DIM_PAD] = PAD_C
        for ki, k in enumerate(K_FREQS):
            theta = 2.0 * np.pi * k * t / P
            W_E[t, ADD_COS_BASE + ki] = float(np.cos(theta))
            W_E[t, ADD_SIN_BASE + ki] = float(np.sin(theta))
    W_E[P,   DIM_PAD] = PAD_C; W_E[P,   DIM_OP] = 1.0
    W_E[P+1, DIM_PAD] = PAD_C; W_E[P+1, DIM_EQ] = 1.0
    sd['tok_emb.weight'] = W_E

    # pos_emb: per-position marker
    W_POS = torch.zeros(SEQ_LEN, D_MODEL)
    for p in range(SEQ_LEN): W_POS[p, POS_DIMS[p]] = POS_M
    sd['pos_emb.weight'] = W_POS

    # Block 0
    sd['blocks.0.norm1.scale'] = torch.ones(D_MODEL)
    sd['blocks.0.norm2.scale'] = torch.ones(D_MODEL)

    # Attention in_proj_weight [3*d_model, d_model] — concat Q | K | V
    W_inproj = torch.zeros(3 * D_MODEL, D_MODEL)
    # Head 0 Q: fires only at eq position
    W_inproj[0, POS_DIMS[3]] = Q_GAIN
    # Head 0 K: monotone in value, large negative on op/eq markers
    W_inproj[D_MODEL + 0, DIM_VAL] = K_GAIN
    W_inproj[D_MODEL + 0, DIM_OP]  = -BIG
    W_inproj[D_MODEL + 0, DIM_EQ]  = -BIG
    # Head 0 V: copies the position's Fourier coefficients
    for ki in range(K):
        W_inproj[2 * D_MODEL + ki,        ADD_COS_BASE + ki] = 1.0
        W_inproj[2 * D_MODEL + K + ki,    ADD_SIN_BASE + ki] = 1.0
    sd['blocks.0.attn.in_proj_weight'] = W_inproj
    sd['blocks.0.attn.in_proj_bias']   = torch.zeros(3 * D_MODEL)

    # Attention out_proj: map head output back into Fourier dims of residual
    W_outproj = torch.zeros(D_MODEL, D_MODEL)
    for ki in range(K):
        W_outproj[ADD_COS_BASE + ki, ki]       = 1.0
        W_outproj[ADD_SIN_BASE + ki, K + ki]   = 1.0
    sd['blocks.0.attn.out_proj.weight'] = W_outproj
    sd['blocks.0.attn.out_proj.bias']   = torch.zeros(D_MODEL)

    # FFN of Block 0 — zero (passthrough)
    sd['blocks.0.w1.weight'] = torch.zeros(4 * D_MODEL, D_MODEL)
    sd['blocks.0.w2.weight'] = torch.zeros(D_MODEL, 4 * D_MODEL)
    sd['blocks.0.w3.weight'] = torch.zeros(4 * D_MODEL, D_MODEL)

    # Block 1 — fully zero (passthrough)
    sd['blocks.1.norm1.scale'] = torch.ones(D_MODEL)
    sd['blocks.1.norm2.scale'] = torch.ones(D_MODEL)
    sd['blocks.1.attn.in_proj_weight'] = torch.zeros(3 * D_MODEL, D_MODEL)
    sd['blocks.1.attn.in_proj_bias']   = torch.zeros(3 * D_MODEL)
    sd['blocks.1.attn.out_proj.weight'] = torch.zeros(D_MODEL, D_MODEL)
    sd['blocks.1.attn.out_proj.bias']   = torch.zeros(D_MODEL)
    sd['blocks.1.w1.weight'] = torch.zeros(4 * D_MODEL, D_MODEL)
    sd['blocks.1.w2.weight'] = torch.zeros(D_MODEL, 4 * D_MODEL)
    sd['blocks.1.w3.weight'] = torch.zeros(4 * D_MODEL, D_MODEL)

    # Final norm + Fourier-decoder head
    sd['norm_f.scale'] = torch.ones(D_MODEL)
    W_HEAD = torch.zeros(P, D_MODEL)
    for c in range(P):
        for ki, k in enumerate(K_FREQS):
            theta = 2.0 * np.pi * k * c / P
            W_HEAD[c, ADD_COS_BASE + ki] = float(np.cos(theta))
            W_HEAD[c, ADD_SIN_BASE + ki] = float(np.sin(theta))
    sd['head.weight'] = W_HEAD
    return sd

state_dict = build_handcraft_max()
print(f'Built {len(state_dict)} tensors. Total non-zero parameter count summary:')
for name, t in state_dict.items():
    nz = int((t != 0).sum().item()); total = int(t.numel())
    pct = 100 * nz / total if total else 0
    print(f'  {name:40s} {tuple(t.shape):<25s}  nnz {nz:>6d}/{total} ({pct:5.1f}%)')

In [ ]:
# Cell 4 — load into the model and verify 100% on all 9409 (a, b) pairs
m = GrokModelMax().to(device)
missing, unexpected = m.load_state_dict(state_dict, strict=True)
assert not missing and not unexpected, (missing, unexpected)
m.eval()

all_a = torch.arange(P, device=device).repeat_interleave(P)
all_b = torch.arange(P, device=device).repeat(P)
all_c = torch.maximum(all_a, all_b)
with torch.no_grad():
    preds = m(all_a, all_b).argmax(-1)
hits = (preds == all_c).cpu().numpy().reshape(P, P)
acc = float(hits.mean())
print(f'Accuracy: {int(hits.sum())} / {P*P} = {acc:.6f}')
assert acc == 1.0, f'Expected 100% accuracy, got {acc}'

torch.save(state_dict, OUT_DIR / 'handcraft_max_state_dict.pt')
with open(OUT_DIR / 'handcraft_max_verification.json', 'w') as f:
    json.dump({'n_pairs': int(P*P), 'n_correct': int(hits.sum()), 'accuracy': acc}, f, indent=2)
print(f'Saved handcraft_max_state_dict.pt and handcraft_max_verification.json')

In [ ]:
# Cell 5 — Figure 1: Fourier circles in tok_emb for value tokens
tok_emb = state_dict['tok_emb.weight'].numpy()
VALUE_TOKENS = list(range(P))
OP_TOKEN, EQ_TOKEN = P, P + 1
K_TO_SHOW = [1, 2, 3, 6, 12, 24]

fig, axes = plt.subplots(1, len(K_TO_SHOW), figsize=(3.0 * len(K_TO_SHOW), 3.2), dpi=120)
for ax, k in zip(axes, K_TO_SHOW):
    cos_dim = ADD_COS_BASE + (k - 1)
    sin_dim = ADD_SIN_BASE + (k - 1)
    xs = tok_emb[VALUE_TOKENS, cos_dim]
    ys = tok_emb[VALUE_TOKENS, sin_dim]
    colors = np.array(VALUE_TOKENS) / (P - 1)
    ax.scatter(xs, ys, c=colors, cmap='twilight', s=22, edgecolor=FG, linewidth=0.3, zorder=3)
    ax.scatter(tok_emb[OP_TOKEN, cos_dim], tok_emb[OP_TOKEN, sin_dim],
               marker='X', color='#D45A2A', s=110, edgecolor=FG, linewidth=0.6, zorder=5, label='op')
    ax.scatter(tok_emb[EQ_TOKEN, cos_dim], tok_emb[EQ_TOKEN, sin_dim],
               marker='P', color='#5E9C5C', s=110, edgecolor=FG, linewidth=0.6, zorder=5, label='eq')
    ax.set_aspect('equal'); ax.set_title(f'k = {k}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.legend(fontsize=7, frameon=False, loc='upper right')
plt.suptitle('Fourier circles in tok_emb: value tokens 0..96 sit on (cos k·v, sin k·v); op and eq are off-circle', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Figure 2: weight-norm map across tensors (which components carry signal)
rows = [(n, float(np.linalg.norm(t.numpy())), int(t.numel())) for n, t in state_dict.items()]
rows.sort(key=lambda r: -r[1])
names = [r[0] for r in rows]; norms = [r[1] for r in rows]
rms = [n / np.sqrt(s) for n, _, s in zip(norms, [None]*len(rows), [r[2] for r in rows])]

fig, axes = plt.subplots(1, 2, figsize=(15, max(4, 0.32 * len(names))), dpi=120, sharey=True)
axes[0].barh(np.arange(len(names)), norms, color='#3A6E8C')
axes[0].set_yticks(np.arange(len(names))); axes[0].set_yticklabels(names, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel('Frobenius norm ||W||'); axes[0].set_title('Total signal per tensor', fontsize=10)
axes[1].barh(np.arange(len(names)), rms, color='#D45A2A')
axes[1].set_xlabel('RMS = ||W|| / sqrt(numel)'); axes[1].set_title('Average per-element magnitude', fontsize=10)
plt.suptitle('Where the signal lives in the handcrafted weights', fontsize=11, y=1.0)
plt.tight_layout()
plt.show()

## What this proves

100% accuracy on all 9409 input pairs from weights placed entirely by formula. No optimizer was used.

- **Value tokens lie on perfect Fourier circles** for every frequency k=1..48 (the bloom equivalent for the handcrafted model).
- **Signal is concentrated in four tensors**: `tok_emb`, `pos_emb`, the max-selector head's Q/K/V (`blocks.0.attn.in_proj_weight`), and the Fourier-decoder `head`. Everything else is zero — Block 1 and all FFNs are passthrough.

This is the strongest mechanistic-interpretability claim available for a given mechanism: not just a description of what the model does, but a model built from understanding alone that does the same thing.